## Cell 1: Setup
**What this demonstrates**: Environment initialisation for a LangGraph ReAct agent — Anthropic as the reasoning model, OpenAI for embeddings, LangGraph for the stateful reasoning loop.
**Expected output**: Setup confirmation with model name, max steps, and tool list.

In [ ]:
%pip install -q langchain langchain-openai langchain-community langchain-anthropic langgraph chromadb python-dotenv

import os
import time
import pathlib
from collections import Counter
from typing import TypedDict, Annotated

from dotenv import load_dotenv, find_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END, add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# Single model for the full reasoning loop — Sonnet has the best tool-use quality
AGENT_MODEL = 'claude-sonnet-4-6'
EMBED_MODEL  = 'text-embedding-3-small'
MAX_STEPS    = 5  # hard cap on reasoning iterations — prevents runaway loops

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
# temperature=0 ensures consistent tool selection in demos
llm = ChatAnthropic(model=AGENT_MODEL, temperature=0)

print('Setup complete')
print(f'  Agent model : {AGENT_MODEL}')
print(f'  Embed model : {EMBED_MODEL}')
print(f'  Max steps   : {MAX_STEPS}')
print('  Tools       : retrieve_docs | web_search | calculate  (defined in Cell 3)')

## Cell 2: Data — Unified Knowledge Base
**What this demonstrates**: Loading all four sample documents into a single Chroma vector store — the agent's internal retrieval corpus. The multi-source setup (policy + regulation + derivatives + earnings) mirrors a real fintech knowledge base.
**Expected output**: Chunk count per document and corpus summary.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

DOC_FILES = {
    'fintech_policy' : 'fintech_policy.txt',
    'basel_iii'      : 'basel_iii_excerpt.txt',
    'isda'           : 'isda_excerpt.txt',
    'earnings_report': 'earnings_report.txt',
}

raw_docs: list[Document] = []
for source_name, filename in DOC_FILES.items():
    text = (BASE_DIR / filename).read_text(encoding='utf-8')
    raw_docs.append(Document(page_content=text, metadata={'source': source_name}))
    print(f'  {source_name:<18}: {len(text):,} chars')

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks: list[Document] = splitter.split_documents(raw_docs)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='agentic_rag_corpus',
)

print(f'\nCorpus : {len(chunks)} chunks across {len(raw_docs)} documents')
print('Index  : Chroma ready')
print('Note   : web_search uses mock regulatory data — set TAVILY_API_KEY for live results')

## Cell 3: Core — Tools, Agent, Reasoning Loop
**What this demonstrates**: Three tools (`retrieve_docs`, `web_search`, `calculate`) registered in a LangGraph `StateGraph`. The agent reasons at each step, chooses a tool, observes the result, and continues until evidence is complete or `MAX_STEPS` fires.
**Expected output**: Pipeline confirmation with tool names and graph structure.

In [ ]:
# ── Agent state ───────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # full message history, appended each step
    steps: int                               # iteration counter for the stopping condition


# ── Tool definitions ──────────────────────────────────────────────────────────

@tool
def retrieve_docs(query: str) -> str:
    'Search the internal corpus for regulatory requirements, policies, and financial data. Use for: Basel III rules, lending policy, ISDA terms, earnings figures.'
    docs = vectorstore.similarity_search(query, k=4)
    if not docs:
        return 'No relevant documents found in internal corpus.'
    parts = []
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get('source', 'unknown')
        parts.append(f'[Doc {i} — {src}]\n{doc.page_content}')
    return '\n\n---\n\n'.join(parts)


@tool
def web_search(query: str) -> str:
    'Search for current regulatory updates beyond the internal corpus. Use for: recent rule changes, current capital ratio benchmarks, latest regulatory guidance.'
    _key = os.environ.get('TAVILY_API_KEY', '')
    if _key:
        try:
            from tavily import TavilyClient
            res = TavilyClient(api_key=_key).search(query=query, max_results=3)
            parts = []
            for r in res.get('results', []):
                title   = r.get('title', '')
                content = r.get('content', '')
                parts.append(f'[{title}]\n{content}')
            return '\n\n'.join(parts)
        except Exception:
            pass  # fall through to mock
    # Mock regulatory data — replace with Tavily in production
    return '\n'.join([
        'Regulatory updates (mock — set TAVILY_API_KEY for live results):',
        '- Basel IV (Jan 2025): Output floor of 72.5% on internal model RWA now in effect',
        '- BCBS minimums: CET1 4.5%, Tier 1 6%, Total capital 8% of RWA',
        '- G-SIB surcharges: 1%-3.5% additional CET1 by systemic importance bucket',
        '- SA-CCR (counterparty credit risk): effective Q1 2025',
        '- EBA 2024 stress test: average CET1 across EU banks at 15.6%',
        'Source: BIS.org regulatory calendar (mock data)',
    ])


@tool
def calculate(expression: str) -> str:
    'Evaluate arithmetic for portfolio values and capital ratios. Supports +, -, *, /, parentheses. Example: 500000000 * 0.08 returns the 8% capital charge on a $500M portfolio.'
    safe_chars = set('0123456789 +-*/.() ')
    if not set(expression).issubset(safe_chars):
        return f'Cannot evaluate — only numeric expressions supported'
    try:
        result = eval(expression, {'__builtins__': {}}, {})  # noqa: S307
        return f'{expression} = {result:,.2f}'
    except Exception as e:
        return f'Calculation error: {e}'


TOOLS = [retrieve_docs, web_search, calculate]
llm_with_tools = llm.bind_tools(TOOLS)


# ── Agent node ────────────────────────────────────────────────────────────────

def call_model(state: AgentState) -> dict:
    'One reasoning step: LLM decides to call a tool or produce a final answer.'
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response], 'steps': state['steps'] + 1}


# ── Stopping condition ────────────────────────────────────────────────────────

def should_continue(state: AgentState) -> str:
    'Continue to tools, or end. Ends when no tool_calls or step cap reached.'
    last = state['messages'][-1]
    if state['steps'] >= MAX_STEPS:
        return '__end__'  # hard stop — prevents infinite loops
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    return '__end__'


# ── Build LangGraph StateGraph ────────────────────────────────────────────────

workflow = StateGraph(AgentState)
workflow.add_node('agent', call_model)
workflow.add_node('tools', ToolNode(TOOLS))
workflow.set_entry_point('agent')
workflow.add_conditional_edges(
    'agent',
    should_continue,
    {'tools': 'tools', '__end__': END},
)
workflow.add_edge('tools', 'agent')  # tool results always feed back to the agent
agent = workflow.compile()

print('Agentic RAG pipeline defined')
print(f"  Tools     : {[t.name for t in TOOLS]}")
print(f'  Max steps : {MAX_STEPS}')
print('  Graph     : START → agent → (tool_calls?) ─┬→ tools → agent → ...')
print('                                              └→ END')

## Cell 4: Run — Basel III Capital Requirement Query
**What this demonstrates**: A query requiring three tools in sequence — internal retrieval (Basel III rules), web search (current regulatory updates), and calculation ($500M × 8%). The agent determines the tool sequence at runtime.
**Expected output**: Steps taken, tools called, and a grounded final answer with source attribution.

In [ ]:
SYSTEM_PROMPT = '\n'.join([
    'You are a financial regulatory analyst with access to three tools:',
    '  retrieve_docs  — search the internal document corpus',
    '  web_search     — find current regulatory updates beyond the corpus',
    '  calculate      — evaluate arithmetic expressions',
    'For each query: retrieve relevant documents, search for current updates if needed,',
    'use calculate for any numerical computations, then synthesise a complete answer.',
    'Cite the source (document name or web search) for each specific claim.',
])

QUERY = (
    'What is the total capital requirement for a $500M loan portfolio under Basel III, '
    'and what are the current regulatory updates?'
)

print(f'Query: {QUERY}')
print()
print('Running agent...')
print()

initial_state: AgentState = {
    'messages': [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=QUERY),
    ],
    'steps': 0,
}

t0 = time.perf_counter()
final_state = agent.invoke(initial_state)
elapsed_ms  = (time.perf_counter() - t0) * 1000

print(f'Completed in {elapsed_ms:.0f} ms  |  {final_state["steps"]} reasoning steps')
print()
print('=' * 65)
print('FINAL ANSWER')
print('=' * 65)

final_msg = final_state['messages'][-1]
answer = final_msg.content
# Claude may return a list of content blocks — extract text
if isinstance(answer, list):
    answer = ' '.join(
        block.get('text', '') if isinstance(block, dict) else str(block)
        for block in answer
    )
print(answer)

## Cell 5: Inspect — Agent Reasoning Trace
**What this demonstrates**: The full message history — every reasoning step, tool call, and observation. This is the audit trail that makes the agent's decisions transparent and debuggable in production.
**Expected output**: Annotated message sequence, tool call summary, and production monitoring guidance.

In [ ]:
print('AGENT REASONING TRACE')
print('=' * 65)

tool_calls_made: list[str] = []
messages = final_state['messages']

for i, msg in enumerate(messages):
    msg_type = type(msg).__name__

    if msg_type == 'SystemMessage':
        print(f'[{i}] SystemMessage  — system prompt ({len(str(msg.content))} chars)')

    elif msg_type == 'HumanMessage':
        preview = str(msg.content)[:60].replace('\n', ' ')
        print(f'[{i}] HumanMessage   — {preview}...')

    elif msg_type == 'AIMessage':
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls_made.append(tc['name'])
                args_preview = str(tc.get('args', {}))[:55]
                print(f'[{i}] AIMessage      → TOOL CALL : {tc["name"]}({args_preview})')
        else:
            content = msg.content
            if isinstance(content, list):
                content = ' '.join(b.get('text', '') if isinstance(b, dict) else str(b) for b in content)
            preview = str(content)[:70].replace('\n', ' ')
            print(f'[{i}] AIMessage      → FINAL ANSWER: {preview}...')

    elif msg_type == 'ToolMessage':
        preview = str(msg.content)[:70].replace('\n', ' ')
        print(f'[{i}] ToolMessage    ← OBSERVATION : {preview}...')

# ── Tool call summary ─────────────────────────────────────────────────────────
print()
print('─' * 65)
print('TOOL CALL SUMMARY')
for tool_name, count in Counter(tool_calls_made).items():
    print(f'  {tool_name:<25}: {count} call(s)')
print(f'  {"Total steps used":<25}: {final_state["steps"]} / {MAX_STEPS}')
print(f'  {"Step cap hit":<25}: {final_state["steps"] >= MAX_STEPS}')
print()
print('In production: log tool_calls_made and steps per query.')
print('A query hitting max_steps signals a loop or a missing tool description.')
print('A query using only retrieve_docs is a candidate for tier-1 routing in Adaptive RAG.')

## Cell 6: Fintech — Multi-Source Compliance Research
**What this demonstrates**: The workshop demo query — a capital adequacy compliance check requiring internal retrieval, ratio calculation, and current regulatory updates. The agent determines all three are needed without being told.
**Expected output**: Tool call sequence, compliance analysis, and a summary showing why no static pipeline handles this query.

In [ ]:
COMPLIANCE_QUERY = '\n'.join([
    'Analyse Basel III capital adequacy for a bank with the following profile:',
    '  Total RWA: $2.8B',
    '  CET1 capital: $280M',
    '  Tier 1 capital: $350M',
    '  Total capital: $420M',
    'Calculate the current CET1, Tier 1, and Total capital ratios.',
    'Check against Basel III minimums.',
    'Include any current regulatory updates relevant to these requirements.',
])

print('MULTI-SOURCE COMPLIANCE RESEARCH')
print(f'Query:\n  {COMPLIANCE_QUERY.replace(chr(10), chr(10) + "  ")}')
print()
print('Running agent...')

compliance_state: AgentState = {
    'messages': [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=COMPLIANCE_QUERY),
    ],
    'steps': 0,
}

t0 = time.perf_counter()
compliance_result = agent.invoke(compliance_state)
elapsed_ms = (time.perf_counter() - t0) * 1000

# Extract tool call sequence
tool_sequence: list[str] = []
for msg in compliance_result['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            tool_sequence.append(tc['name'])

print(f'Tool sequence : {" → ".join(tool_sequence)}')
print(f'Steps used    : {compliance_result["steps"]} / {MAX_STEPS}')
print(f'Elapsed       : {elapsed_ms:.0f} ms')
print()
print('=' * 65)
print('COMPLIANCE ANALYSIS')
print('=' * 65)

final_msg = compliance_result['messages'][-1]
answer = final_msg.content
if isinstance(answer, list):
    answer = ' '.join(
        block.get('text', '') if isinstance(block, dict) else str(block)
        for block in answer
    )
print(answer)
print()
print('─' * 65)
print('WHY A STATIC PIPELINE CANNOT HANDLE THIS:')
print('  retrieve_docs  — Basel III minimums from internal corpus')
print('  calculate      — ratio verification (CET1%, Tier1%, Total capital%)')
print('  web_search     — current regulatory updates beyond the corpus')
print('The agent determined the tool sequence and step count at runtime.')
print('No pre-defined pipeline knows in advance which tools this query needs.')